In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s4e6/sample_submission.csv
/kaggle/input/competitions/playground-series-s4e6/train.csv
/kaggle/input/competitions/playground-series-s4e6/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s4e6/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s4e6/test.csv")
sample= pd.read_csv("/kaggle/input/competitions/playground-series-s4e6/sample_submission.csv")

**Cheak data**
データの確認

In [3]:
train.shape

(76518, 38)

In [4]:
test.shape

(51012, 37)

In [5]:
train.head()

,id,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,0,1,1,1,9238,1,1,126.0,1,1,...,0,6,7,6,12.428571,0,11.1,0.6,2.02,Graduate
1,1,1,17,1,9238,1,1,125.0,1,19,...,0,6,9,0,0.000000,0,11.1,0.6,2.02,Dropout
2,2,1,17,2,9254,1,1,137.0,1,3,...,0,6,0,0,0.000000,0,16.2,0.3,-0.92,Dropout
3,3,1,1,3,9500,1,1,131.0,1,19,...,0,8,11,7,12.820000,0,11.1,0.6,2.02,Enrolled
4,4,1,1,2,9500,1,1,132.0,1,19,...,0,7,12,6,12.933333,0,7.6,2.6,0.32,Graduate


In [6]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76518 entries, 0 to 76517
Data columns (total 38 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   id                                              76518 non-null  int64  
 1   Marital status                                  76518 non-null  int64  
 2   Application mode                                76518 non-null  int64  
 3   Application order                               76518 non-null  int64  
 4   Course                                          76518 non-null  int64  
 5   Daytime/evening attendance                      76518 non-null  int64  
 6   Previous qualification                          76518 non-null  int64  
 7   Previous qualification (grade)                  76518 non-null  float64
 8   Nacionality                                     76518 non-null  int64  
 9   Mother's qualification                 

In [7]:
train["Target"].unique()

array(['Graduate', 'Dropout', 'Enrolled'], dtype=object)

**Checking for missing values**
欠損値の確認

In [8]:
train.isnull().sum()

id                                                0
Marital status                                    0
Application mode                                  0
Application order                                 0
Course                                            0
Daytime/evening attendance                        0
Previous qualification                            0
Previous qualification (grade)                    0
Nacionality                                       0
Mother's qualification                            0
Father's qualification                            0
Mother's occupation                               0
Father's occupation                               0
Admission grade                                   0
Displaced                                         0
Educational special needs                         0
Debtor                                            0
Tuition fees up to date                           0
Gender                                            0
Scholarship 

In [9]:
test.isnull().sum()

id                                                0
Marital status                                    0
Application mode                                  0
Application order                                 0
Course                                            0
Daytime/evening attendance                        0
Previous qualification                            0
Previous qualification (grade)                    0
Nacionality                                       0
Mother's qualification                            0
Father's qualification                            0
Mother's occupation                               0
Father's occupation                               0
Admission grade                                   0
Displaced                                         0
Educational special needs                         0
Debtor                                            0
Tuition fees up to date                           0
Gender                                            0
Scholarship 

**Creation of test data**
テストデータの作成

In [10]:
X_train = train.drop(columns=["id", "Target"])

In [11]:
y_train = train["Target"]

In [12]:
X_test = test.drop(columns=["id"])

In [13]:
X_train["Total approved"] = (
    X_train["Curricular units 1st sem (approved)"] +
    X_train["Curricular units 2nd sem (approved)"]
)

X_test["Total approved"] = (
    X_test["Curricular units 1st sem (approved)"] +
    X_test["Curricular units 2nd sem (approved)"]
)

In [14]:
X_train["Mean grade"] = (
    X_train["Curricular units 1st sem (grade)"] +
    X_train["Curricular units 2nd sem (grade)"]
) / 2

X_test["Mean grade"] = (
    X_test["Curricular units 1st sem (grade)"] +
    X_test["Curricular units 2nd sem (grade)"]
) / 2

In [15]:
X_train["Total evaluations"] = (
    X_train["Curricular units 1st sem (evaluations)"] +
    X_train["Curricular units 2nd sem (evaluations)"]
)

X_test["Total evaluations"] = (
    X_test["Curricular units 1st sem (evaluations)"] +
    X_test["Curricular units 2nd sem (evaluations)"]
)

In [16]:
from sklearn.model_selection import train_test_split

X_train2, X_valid, y_train2, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

**cheak**
確認

In [17]:
X_train.shape

(76518, 39)

In [18]:
y_train.shape

(76518,)

In [19]:
X_test.shape

(51012, 39)

**Random Forest**
アルゴリズムのライブラリを読み込み

In [20]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=5,
    random_seed=42,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    verbose=200
)

In [21]:
model.fit(
    X_train2,
    y_train2,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

0:	learn: 0.8053877	test: 0.8013591	best: 0.8013591 (0)	total: 150ms	remaining: 7m 28s
200:	learn: 0.8291567	test: 0.8250131	best: 0.8250784 (198)	total: 18.4s	remaining: 4m 16s
400:	learn: 0.8364100	test: 0.8283455	best: 0.8284762 (395)	total: 36.6s	remaining: 3m 57s
600:	learn: 0.8425197	test: 0.8290643	best: 0.8293257 (579)	total: 54.8s	remaining: 3m 38s
800:	learn: 0.8481883	test: 0.8291296	best: 0.8293257 (579)	total: 1m 12s	remaining: 3m 20s
1000:	learn: 0.8535792	test: 0.8295217	best: 0.8305018 (897)	total: 1m 31s	remaining: 3m 1s
1200:	learn: 0.8585291	test: 0.8306979	best: 0.8309592 (1179)	total: 1m 49s	remaining: 2m 43s
1400:	learn: 0.8632339	test: 0.8307632	best: 0.8312859 (1322)	total: 2m 7s	remaining: 2m 24s
1600:	learn: 0.8676447	test: 0.8308939	best: 0.8312859 (1322)	total: 2m 25s	remaining: 2m 6s
1800:	learn: 0.8712386	test: 0.8303711	best: 0.8312859 (1322)	total: 2m 43s	remaining: 1m 48s
2000:	learn: 0.8755677	test: 0.8295870	best: 0.8312859 (1322)	total: 3m 1s	remaini

CatBoostClassifier(depth=8, eval_metric='Accuracy', iterations=3000, l2_leaf_reg=5, learning_rate=0.03, loss_function='MultiClass', random_seed=42, verbose=200)

In [22]:
valid_pred = model.predict(X_valid)

In [23]:
from sklearn.metrics import accuracy_score

score = accuracy_score(y_valid, valid_pred)
print(score)

0.8322007318348145


In [24]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print(importance.head(15))

                                   Feature  Importance
30     Curricular units 2nd sem (approved)    7.800298
36                          Total approved    6.933985
3                                   Course    5.908362
12                         Admission grade    5.874654
31        Curricular units 2nd sem (grade)    5.586459
6           Previous qualification (grade)    4.591385
37                              Mean grade    4.347247
19                       Age at enrollment    4.230615
38                       Total evaluations    4.148503
29  Curricular units 2nd sem (evaluations)    4.043153
25        Curricular units 1st sem (grade)    3.688329
11                     Father's occupation    3.677570
16                 Tuition fees up to date    3.312311
9                   Father's qualification    2.935626
23  Curricular units 1st sem (evaluations)    2.819045


In [25]:
pred = model.predict(X_test)

In [26]:
pred = pred.ravel()

In [27]:
pred[:10]

array(['Dropout', 'Graduate', 'Graduate', 'Graduate', 'Enrolled',
       'Graduate', 'Graduate', 'Graduate', 'Dropout', 'Graduate'],
      dtype=object)

In [28]:
sample.head()

,id,Target
0,76518,Graduate
1,76519,Graduate
2,76520,Graduate
3,76521,Graduate
4,76522,Graduate


In [29]:
submission = sample.copy()

submission["Target"] = pred

submission.to_csv("submission.csv", index=False)

In [30]:
submission = sample.copy()

submission["Target"] = pred

submission.to_csv("submission.csv", index=False)

submission.head()

,id,Target
0,76518,Dropout
1,76519,Graduate
2,76520,Graduate
3,76521,Graduate
4,76522,Enrolled
